## Prep for text download

Load libraries

In [1]:
import datetime as dt
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import mediacloud.api as mcapi
from pathlib import Path
from tqdm import tqdm
from bs4 import BeautifulSoup
from newspaper import Article 
import os
import math

Load fixed parameters

In [2]:
# Set your personal API KEY
MC_API_KEY = 'b32652584a00c2bc643b93fe210c9ade8fd6df80'
mc = mcapi.SearchApi(MC_API_KEY)

# Load data
OUTPUT_PATH = Path(r"C:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Mediacloud\Mediacloud_download+basic_graphs")
CSV_PATH  = r"c:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Datasets\Google_dataset.csv"
ENCODING = "cp1252"
counts_df = pd.read_csv(OUTPUT_PATH / "subtype_queries_daily.csv", header=[0,1,2], index_col=0)
counts_df.index = pd.to_datetime(counts_df.index)
START = dt.date(2010, 1, 1)
END = dt.date(2025, 12, 31)

df = pd.read_csv(CSV_PATH, encoding=ENCODING)
df.columns = df.columns.str.strip()
print(df.head())

SUBTYPE_COLOURS: dict[str, str] = {
    "Aedes":       "#E69F00",
    "Chikungunya": "#56B4E9",
    "Dengue":      "#009E73",
    "Zika":        "#F0E442",
    "Culex":       "#0072B2",
    "Usutu":       "#D55E00",
    "West Nile":   "#CC79A7",
}

SYSTEM_COLOURS: dict[str, str] = {
    "Aedes-borne disease system": "#0072B2",
    "Culex-borne disease system": "#D55E00",
}

SYSTEM_STYLES: dict[str, str] = {
    "Aedes-borne disease system": "dashed",
    "Culex-borne disease system": "solid",
}

COUNTRY_LABELS: dict[str, str] = {
    "DE": ["Germany"],
    "AT": ["Austria"],
    "CH": ["Switzerland"],
    "IT": ["Italy"],
    "ES": ["Spain"],
    "FR": ["France"],
}

COUNTRY_LANGUAGES: dict[str, list[str]] = {
    "DE": ["German"],
    "AT": ["German"],
    "CH": ["German", "French", "Italian"],
    "IT": ["Italian"],
    "ES": ["Spanish"],
    "FR": ["French"],
}

# Preferred country order in the figure
COUNTRY_ORDER = ["Germany", "Austria", "Switzerland", "Italy", "Spain", "France"]

ANCHOR_PER_COUNTRY: dict[str, str] = {
    "Germany": "Mücke",
    "Austria": "Mücke",
    "Switzerland": "Mücke",
    "Italy": "Zanzara",
    "Spain": "Mosquito",
    "France": "Moustique",
}

# Define country filters
country_filters = {
    "Germany": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34412409, 38379816],
    },
    "Austria": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34412245, 38378035],
    },
    "Switzerland": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34411591, 38380954],
    },
    "Italy": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34412372, 38380117],
    },
    "Spain": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34412356, 38002034],
    },
    "France": {
        "start_date": START,
        "end_date": END,
        "collection_ids": [34412146, 38379799],
    },
}

                Keyword                      System Sub-type     Type Language
0      Aedes albopictus  Aedes-borne disease system    Aedes   Animal    Latin
1        Aedes mosquito  Aedes-borne disease system    Aedes   Animal  English
2  Asian tiger mosquito  Aedes-borne disease system    Aedes   Animal  English
3        Tiger mosquito  Aedes-borne disease system    Aedes   Animal  English
4                  Zika  Aedes-borne disease system     Zika  Disease      All


Make Solr search strings for url download (per subtype only)

In [3]:
# Sub-type lists per country
country_subtype_kws = {}

for geo, languages in COUNTRY_LANGUAGES.items():

    country_name = COUNTRY_LABELS[geo][0]

    country_df = df[df["Language"].isin(languages)]

    country_subtype_kws[country_name] = {}

    for subtype, group in country_df.groupby("Sub-type"):

        kws = group["Keyword"].tolist()

        universal = df.loc[
            (df["Language"].isin(["All", "Latin"])) &
            (df["Sub-type"] == subtype),
            "Keyword"
        ].tolist()

        country_subtype_kws[country_name][subtype] = list(
            dict.fromkeys(kws + universal)
        )

print(country_subtype_kws)

# Restructure dictionary into Solr query strings
def build_mc_query(keywords):
    kw_part = " OR ".join([f'"{k}"' for k in keywords])
    query = f"({kw_part})"
    return query

subtype_queries_temp = {}

for country, subtypes in country_subtype_kws.items():
    for subtype, keywords in subtypes.items():
        subtype_queries_temp[(country, subtype)] = build_mc_query(keywords)

{'Germany': {'Aedes': ['Aedes-Mücke', 'Aedes Mücke', 'Tigermücke', 'Tiger Mücke', 'Asiatische Tigermücke', 'Aedes albopictus'], 'Chikungunya': ['Chikungunyafieber', 'Chikungunya Fieber', 'Chikungunya-Virus', 'Chikungunya Virus', 'Chikungunya'], 'Culex': ['Gemeine Stechmücke', 'Nördliche Hausmücke', 'Hausmücke', 'Culex-Mücke', 'Culex pipiens', 'Culex torrentium', 'Culex modestus'], 'Dengue': ['Denguefieber', 'Dengue-Fieber', 'Dengue Fieber', 'Knochenbrecherfieber', 'Siebentagefieber', 'Dengue'], 'Usutu': ['Usutu-Virus', 'Usutu Virus'], 'West Nile': ['West-Nil-Virus', 'West Nil Virus', 'West-Nil-Fieber', 'West Nil Fieber'], 'Zika': ['Zika Virus', 'Zikavirus', 'Zika-Virus', 'Zika-Fieber', 'Zika Fieber', 'Zika']}, 'Austria': {'Aedes': ['Aedes-Mücke', 'Aedes Mücke', 'Tigermücke', 'Tiger Mücke', 'Asiatische Tigermücke', 'Aedes albopictus'], 'Chikungunya': ['Chikungunyafieber', 'Chikungunya Fieber', 'Chikungunya-Virus', 'Chikungunya Virus', 'Chikungunya'], 'Culex': ['Gemeine Stechmücke', 'Nör

Retrieve URLs to get full texts

In [4]:
# Estimate unique stories per (country, subtype) under Option 3
# Assumptions: 6 calls/year × 10 stories = 60 max, but deduplicated against total available stories

results = []

for (country, subtype, metric), series in counts_df.items():
    if metric == "count":
        total_stories = series.sum()  # total stories across all years
        years_with_stories = sum(1 for date, val in series.items() if pd.notna(val) and val > 0)
        max_sampled = years_with_stories * 6 * 10  # 6 calls × 10 stories × years with hits
        # can't sample more than exist
        estimated_unique = min(max_sampled, total_stories)
        results.append({
            "country": country,
            "subtype": subtype,
            "total_stories_in_corpus": int(total_stories),
            "years_with_stories": years_with_stories,
            "max_sample": max_sampled,
            "estimated_unique": int(estimated_unique),
        })

results_df = pd.DataFrame(results).sort_values(["country", "subtype"])
print(results_df.to_string(index=False))
print(f"\nTotal estimated unique stories: {results_df['estimated_unique'].sum():,}")

viable_pairs = results_df[results_df["estimated_unique"] >= 100]
print(viable_pairs)


    country     subtype  total_stories_in_corpus  years_with_stories  max_sample  estimated_unique
    Austria       Aedes                      561                 312       18720               561
    Austria Chikungunya                      495                 266       15960               495
    Austria       Culex                       93                  51        3060                93
    Austria      Dengue                     1209                 630       37800              1209
    Austria       Usutu                       74                  37        2220                74
    Austria   West Nile                      448                 242       14520               448
    Austria        Zika                     1659                 546       32760              1659
     France       Aedes                     8809                2298      137880              8809
     France Chikungunya                     9490                2439      146340              9490
     Franc

In [5]:
# Check if i have enough api calls available
pairs = [
    # (country, subtype, estimated_unique)
    ("Austria", "Aedes", 561),
    ("Austria", "Chikungunya", 495),
    ("Austria", "Culex", 93),
    ("Austria", "Dengue", 1209),
    ("Austria", "Usutu", 74),
    ("Austria", "West Nile", 448),
    ("Austria", "Zika", 1659),
    ("France", "Aedes", 8809),
    ("France", "Chikungunya", 9490),
    ("France", "Culex", 494),
    ("France", "Dengue", 12527),
    ("France", "Usutu", 110),
    ("France", "West Nile", 1429),
    ("France", "Zika", 11225),
    ("Germany", "Aedes", 7825),
    ("Germany", "Chikungunya", 4793),
    ("Germany", "Culex", 970),
    ("Germany", "Dengue", 12726),
    ("Germany", "Usutu", 1243),
    ("Germany", "West Nile", 5154),
    ("Germany", "Zika", 11196),
    ("Italy", "Aedes", 3041),
    ("Italy", "Chikungunya", 2373),
    ("Italy", "Culex", 853),
    ("Italy", "Dengue", 5739),
    ("Italy", "Usutu", 149),
    ("Italy", "West Nile", 605),
    ("Italy", "Zika", 4075),
    ("Spain", "Aedes", 13334),
    ("Spain", "Chikungunya", 5922),
    ("Spain", "Culex", 2262),
    ("Spain", "Dengue", 26152),
    ("Spain", "Usutu", 175),
    ("Spain", "West Nile", 6870),
    ("Spain", "Zika", 22479),
    ("Switzerland", "Aedes", 1346),
    ("Switzerland", "Chikungunya", 851),
    ("Switzerland", "Culex", 96),
    ("Switzerland", "Dengue", 2123),
    ("Switzerland", "Usutu", 36),
    ("Switzerland", "West Nile", 247),
    ("Switzerland", "Zika", 3481),
]

LIST_THRESHOLD = 2700
SAMPLE_CALLS = 90  # max calls for story_sample pairs
remaining_quota = 4000 - 382  # = 3618

list_pairs = []
sample_pairs = []

for country, subtype, est in pairs:
    if est < LIST_THRESHOLD:
        calls = math.ceil(est / 100)
        list_pairs.append((country, subtype, est, calls))
    else:
        sample_pairs.append((country, subtype, est, SAMPLE_CALLS))

total_list_calls = sum(c for *_, c in list_pairs)
total_sample_calls = sum(c for *_, c in sample_pairs)
total_calls = total_list_calls + total_sample_calls

print(f"story_list  pairs: {len(list_pairs)}, calls: {total_list_calls}")
print(f"story_sample pairs: {len(sample_pairs)}, calls: {total_sample_calls}")
print(f"Total calls needed: {total_calls}")
print(f"Remaining quota:    {remaining_quota}")
print(f"Buffer:             {remaining_quota - total_calls}")
print()
print("story_list pairs:")
for country, subtype, est, calls in sorted(list_pairs, key=lambda x: -x[2]):
    print(f"  {country}/{subtype}: est={est}, calls={calls}")
print()
print("story_sample pairs:")
for country, subtype, est, calls in sample_pairs:
    print(f"  {country}/{subtype}: est={est}, calls={calls}")

story_list  pairs: 24, calls: 210
story_sample pairs: 18, calls: 1620
Total calls needed: 1830
Remaining quota:    3618
Buffer:             1788

story_list pairs:
  Italy/Chikungunya: est=2373, calls=24
  Spain/Culex: est=2262, calls=23
  Switzerland/Dengue: est=2123, calls=22
  Austria/Zika: est=1659, calls=17
  France/West Nile: est=1429, calls=15
  Switzerland/Aedes: est=1346, calls=14
  Germany/Usutu: est=1243, calls=13
  Austria/Dengue: est=1209, calls=13
  Germany/Culex: est=970, calls=10
  Italy/Culex: est=853, calls=9
  Switzerland/Chikungunya: est=851, calls=9
  Italy/West Nile: est=605, calls=7
  Austria/Aedes: est=561, calls=6
  Austria/Chikungunya: est=495, calls=5
  France/Culex: est=494, calls=5
  Austria/West Nile: est=448, calls=5
  Switzerland/West Nile: est=247, calls=3
  Spain/Usutu: est=175, calls=2
  Italy/Usutu: est=149, calls=2
  France/Usutu: est=110, calls=2
  Switzerland/Culex: est=96, calls=1
  Austria/Culex: est=93, calls=1
  Austria/Usutu: est=74, calls=1


In [13]:
def fetch_all_stories_list(mc, query, country, subtype, collection_ids,
                            start_date, end_date, target_docs=900, retries=3):
    records = []
    seen_ids = set()
    pagination_token = None
    with tqdm(desc=f"{country}/{subtype} [list]", leave=False) as pbar:
        while True:
            # ── early exit once we have enough ──────────────────────────
            if len(records) >= target_docs:
                break
            # ────────────────────────────────────────────────────────────
            for attempt in range(retries):
                try:
                    page, next_token = mc.story_list(
                        query,
                        collection_ids=collection_ids,
                        start_date=start_date,
                        end_date=end_date,
                        pagination_token=pagination_token,
                        page_size=100
                    )
                    break
                except Exception as e:
                    if attempt < retries - 1:
                        time.sleep(2 ** attempt)
                    else:
                        print(f"  Failed: {e}")
                        return records
            if not page:
                break
            for story in page:
                sid = story.get("id")
                if sid not in seen_ids:
                    seen_ids.add(sid)
                    records.append({
                        "country": country,
                        "subtype": subtype,
                        "story_id": sid,
                        "url": story.get("url"),
                        "title": story.get("title"),
                        "publish_date": story.get("publish_date"),
                        "media_name": story.get("media_name"),
                    })
            pbar.update(len(page))
            if next_token is None:
                break
            pagination_token = next_token
    # No longer needs random.sample since we stopped at target_docs
    return records[:target_docs]


def sample_subtype_quota_safe(mc, query, country, subtype, collection_ids,
                               start_date, end_date, target_docs=900,
                               max_calls = None, retries=3):
    
    if max_calls is None:
        max_calls = min(200, int(target_docs / 5))

    records = []
    seen_ids = set()
    consecutive_empty = 0

    with tqdm(total=max_calls, desc=f"{country}/{subtype} [sample]", leave=False) as pbar:
        for call in range(max_calls):
            if len(records) >= target_docs:
                break

            for attempt in range(retries):
                try:
                    page = mc.story_sample(
                        query,
                        collection_ids=collection_ids,
                        start_date=start_date,
                        end_date=end_date
                    )
                    break
                except Exception:
                    if attempt < retries - 1:
                        time.sleep(2 ** attempt)
                    else:
                        return records

            new_this_call = 0
            for story in page:
                sid = story.get("id")
                if sid not in seen_ids:
                    seen_ids.add(sid)
                    new_this_call += 1
                    records.append({
                        "country": country,
                        "subtype": subtype,
                        "story_id": sid,
                        "url": story.get("url"),
                        "title": story.get("title"),
                        "publish_date": story.get("publish_date"),
                        "media_name": story.get("media_name"),
                    })
            pbar.update(1)

            # Only early-exit if the API returned nothing at all (true empty page),
            # not just because all stories this call were duplicates
            if page is None or len(page) == 0:
                consecutive_empty += 1
                if consecutive_empty >= 3:
                    break
            else:
                consecutive_empty = 0

    return records

In [23]:
## Run sampling (time intensive)
# Load existing stories to avoid re-fetching
existing = pd.read_csv("sampled_subtype_stories.csv", encoding="utf-8-sig")
existing_counts = existing.groupby(["country", "subtype"]).size()
out_path = OUTPUT_PATH / "sampled_subtype_stories.csv"

SMALL_PAIR_THRESHOLD = 900

all_records = existing.to_dict("records")  # start with what you already have

with tqdm(total=len(results_df), desc="Sampling all pairs") as outer_pbar:
    for _, row in results_df.iterrows():
        country = row["country"]
        subtype = row["subtype"]
        estimated = row["estimated_unique"]
        query = subtype_queries_temp[(country, subtype)]
        collection_ids = country_filters[country]["collection_ids"]

        already_have = existing_counts.get((country, subtype), 0)
        target = min(900, estimated)

        if already_have >= 50:
            print(f"[SKIP] {country}/{subtype} — already have {already_have} stories")
            outer_pbar.update(1)
            continue

        print(f"[RUN] {country}/{subtype} (est={estimated}, have={already_have}) ...", flush=True)

        recs = fetch_all_stories_list(
                mc, query=query, country=country, subtype=subtype,
                collection_ids=collection_ids,
                start_date=START, end_date=END, target_docs=900
            )

        print(f"  → got {len(recs)} stories")
        all_records.extend(recs)

        # after all_records.extend(recs), save a checkpoint
        checkpoint_df = (
            pd.DataFrame(all_records)
            .drop_duplicates(subset=["story_id"])
        )
        checkpoint_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        profile = mc.user_profile()
        hits_used = profile['quota']['hits']
        print(f"  [Quota] {hits_used}/4000 hits used this week")
        outer_pbar.update(1)

# Deduplicate and save
base = (
    pd.DataFrame(all_records)
    .drop_duplicates(subset=["story_id"])
    .sort_values(["country", "subtype", "publish_date"])
    .reset_index(drop=True)
)

sampled_df = pd.concat(
    [
        group.sample(n=min(len(group), 900), random_state=42)
        for _, group in base.groupby(["country", "subtype"])
    ],
    ignore_index=True,
)

sampled_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(sampled_df)} stories → {out_path}")

Sampling all pairs:   0%|          | 0/42 [00:00<?, ?it/s]

[SKIP] Austria/Aedes — already have 554 stories
[SKIP] Austria/Chikungunya — already have 255 stories
[RUN] Austria/Culex (est=93, have=12) ...


  → got 93 stories


Sampling all pairs:   7%|▋         | 3/42 [00:17<03:53,  5.98s/it]

  [Quota] 998/4000 hits used this week
[SKIP] Austria/Dengue — already have 688 stories
[SKIP] Austria/Usutu — already have 55 stories
[SKIP] Austria/West Nile — already have 288 stories
[SKIP] Austria/Zika — already have 900 stories
[SKIP] France/Aedes — already have 847 stories
[SKIP] France/Chikungunya — already have 825 stories
[SKIP] France/Culex — already have 484 stories
[SKIP] France/Dengue — already have 818 stories
[SKIP] France/Usutu — already have 64 stories
[SKIP] France/West Nile — already have 900 stories
[SKIP] France/Zika — already have 783 stories
[SKIP] Germany/Aedes — already have 839 stories
[SKIP] Germany/Chikungunya — already have 809 stories
[SKIP] Germany/Culex — already have 897 stories
[SKIP] Germany/Dengue — already have 783 stories
[SKIP] Germany/Usutu — already have 886 stories
[SKIP] Germany/West Nile — already have 720 stories
[SKIP] Germany/Zika — already have 790 stories
[SKIP] Italy/Aedes — already have 697 stories
[SKIP] Italy/Chikungunya — already h

...

  [Quota] 999/4000 hits used this week
[SKIP] Switzerland/Dengue — already have 835 stories
[RUN] Switzerland/Usutu (est=36, have=25) ...


  → got 36 stories


Sampling all pairs: 100%|██████████| 42/42 [02:18<00:00,  3.29s/it]

  [Quota] 1000/4000 hits used this week
[SKIP] Switzerland/West Nile — already have 139 stories
[SKIP] Switzerland/Zika — already have 686 stories


Saved 30597 stories → C:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Mediacloud\Mediacloud_download+basic_graphs\sampled_subtype_stories.csv


## Text download

In [24]:
sampled_df = pd.read_csv("sampled_subtype_stories.csv", encoding="utf-8-sig")

# --- Cache file path ---
CACHE_PATH = "texts_cache.parquet"

def fetch_with_newspaper(url):
    """Primary method: newspaper3k"""
    article = Article(url)
    article.download()
    article.parse()
    if article.text and len(article.text) > 100:  # sanity check
        return article.text
    return None

def fetch_with_requests(url):
    """Fallback method: requests + BeautifulSoup"""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; research scraper)"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    # Remove boilerplate tags
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    # Try common article containers first
    for selector in ["article", "main", ".article-body", ".story-body", "#content"]:
        container = soup.select_one(selector)
        if container:
            text = container.get_text(separator=" ", strip=True)
            if len(text) > 100:
                return text

    # Last resort: all paragraph tags
    paragraphs = soup.find_all("p")
    text = " ".join(p.get_text(strip=True) for p in paragraphs)
    return text if len(text) > 100 else None

def fetch_text(url):
    """Try newspaper3k first, fall back to requests+BS4"""
    try:
        return fetch_with_newspaper(url)
    except Exception:
        pass
    try:
        return fetch_with_requests(url)
    except Exception:
        return None  # Paywall, dead link, bot protection, etc.


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
CACHE_SAVE_EVERY = 100   # save to disk every N articles
LOG_EVERY        = 500   # print a progress snapshot every N articles

# ── Load cache ───────────────────────────────────────────────────────────────
if os.path.exists(CACHE_PATH):
    cache_df    = pd.read_parquet(CACHE_PATH)
    already_done = set(cache_df["story_id"].tolist())
    print(f"Resuming: {len(already_done)} articles already cached")
else:
    cache_df    = pd.DataFrame(columns=["story_id", "text"])
    already_done = set()

# ── Scrape only what's missing ───────────────────────────────────────────────
to_scrape = sampled_df[~sampled_df["story_id"].isin(already_done)].copy()
print(f"Scraping {len(to_scrape)} remaining articles...")

results      = []
fail_count   = 0
success_count = 0

for i, (_, row) in enumerate(tqdm(
    to_scrape.iterrows(),
    total=len(to_scrape),
    desc="Scraping articles"
), start=1):

    text = fetch_text(row["url"])
    results.append({"story_id": row["story_id"], "text": text})

    # Track success/failure
    if text:
        success_count += 1
    else:
        fail_count += 1

    # ── Periodic save — protects against crashes ─────────────────────────
    if i % CACHE_SAVE_EVERY == 0:
        new_df   = pd.DataFrame(results)
        cache_df = pd.concat([cache_df, new_df], ignore_index=True)
        cache_df.to_parquet(CACHE_PATH, index=False)
        results  = []   # clear the buffer — already persisted

    # ── Periodic health log ───────────────────────────────────────────────
    if i % LOG_EVERY == 0:
        rate     = success_count / i
        elapsed  = i * 1.0          # rough seconds (adjust if you time it)
        remaining = len(to_scrape) - i
        print(
            f"\n[{i}/{len(to_scrape)}] "
            f"✓ {success_count} ok  ✗ {fail_count} failed  "
            f"({rate:.1%} success rate)  "
            f"~{remaining} left"
        )

        # ── Early warning: bail if success rate collapses ─────────────────
        if i >= 1000 and rate < 0.05:
            print("⚠️  Success rate below 5% — check fetch_text() or network!")
            # raise RuntimeError("Aborting: too many failures")  # uncomment to hard-stop

    time.sleep(0.5)

# ── Final save for the leftover buffer ───────────────────────────────────────
if results:
    new_df   = pd.DataFrame(results)
    cache_df = pd.concat([cache_df, new_df], ignore_index=True)
    cache_df.to_parquet(CACHE_PATH, index=False)

# ── Merge and report ─────────────────────────────────────────────────────────
sampled_df = sampled_df.merge(cache_df[["story_id", "text"]], on="story_id", how="left")
print(f"\nDone. Success rate: {sampled_df['text'].notna().mean():.1%}")
print(f"Total fetched: {success_count}  Failed: {fail_count}")

Resuming: 30494 articles already cached
Scraping 103 remaining articles...


Scraping articles: 100%|██████████| 103/103 [02:38<00:00,  1.54s/it]



Done. Success rate: 68.8%
Total fetched: 72  Failed: 31


Exception in thread PyrateLimiter's Leaker:
Traceback (most recent call last):
  File "C:\Users\annab\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\annab\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Mediacloud\Mediacloud_download+basic_graphs\.venv\Lib\site-packages\pyrate_limiter\abstracts\bucket.py", line 268, in _run
    asyncio.run(self._leak(self.sync_buckets))
  File "C:\Users\annab\AppData\Local\Programs\Python\Python311\Lib\asyncio\runners.py", line 190, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\annab\AppData\Local\Programs\Python\Python311\Lib\asyncio\runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\annab\Ap

## AMENDMENT - also include anchors

In [7]:
existing = pd.read_csv("sampled_subtype_stories.csv", encoding="utf-8-sig")
existing_counts = existing.groupby(["country", "subtype"]).size()
out_path = OUTPUT_PATH / "sampled_subtype_stories.csv"
TARGET_DOCS = 900
ANCHOR_SUBTYPE_LABEL = "anchor"

all_records = existing.to_dict("records")

for country, anchor_word in ANCHOR_PER_COUNTRY.items():
    subtype = ANCHOR_SUBTYPE_LABEL
    already_have = existing_counts.get((country, subtype), 0)
    target = TARGET_DOCS

    if already_have >= target - 50:
        print(f"[SKIP] {country}/anchor — already have {already_have}")
        continue

    query = f'"{anchor_word}"'
    collection_ids = country_filters[country]["collection_ids"]

    print(f"[RUN] {country}/anchor (have={already_have}, target={target}) ...", flush=True)
    recs = fetch_all_stories_list(
        mc,
        query=query,
        country=country,
        subtype=subtype,
        collection_ids=collection_ids,
        start_date=START,
        end_date=END,
        target_docs=target
    )
    print(f"  → got {len(recs)} stories")
    all_records.extend(recs)

    # Checkpoint after each country
    checkpoint_df = pd.DataFrame(all_records).drop_duplicates(subset=["story_id"])
    checkpoint_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    # Quota check
    profile = mc.user_profile()
    hits_used = profile['quota']['hits']
    print(f"  [Quota] {hits_used}/4000 hits used this week")

# Final dedup and save
final_df = (
    pd.DataFrame(all_records)
    .drop_duplicates(subset=["story_id"])
    .sort_values(["country", "subtype", "publish_date"])
    .reset_index(drop=True)
)
final_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(final_df)} stories → {out_path}")

[RUN] Germany/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 942/4000 hits used this week
[RUN] Austria/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 951/4000 hits used this week
[RUN] Switzerland/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 960/4000 hits used this week
[RUN] Italy/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 969/4000 hits used this week
[RUN] Spain/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 978/4000 hits used this week
[RUN] France/anchor (have=0, target=900) ...


  → got 900 stories
  [Quota] 987/4000 hits used this week
Saved 30494 stories → C:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Mediacloud\Mediacloud_download+basic_graphs\sampled_subtype_stories.csv


In [10]:
sampled_df = pd.read_csv("sampled_subtype_stories.csv", encoding="utf-8-sig")
CACHE_SAVE_EVERY = 100   # save to disk every N articles
LOG_EVERY        = 500   # print a progress snapshot every N articles

CACHE_PATH = "texts_cache.parquet"

# Load cache
if os.path.exists(CACHE_PATH):
    cache_df = pd.read_parquet(CACHE_PATH)
    already_done = set(cache_df["story_id"].tolist())
    print(f"Resuming: {len(already_done)} articles already cached")
else:
    cache_df = pd.DataFrame(columns=["story_id", "text"])
    already_done = set()

# Scrape only missing stories (anchors + any new subtype stories)
to_scrape = sampled_df[~sampled_df["story_id"].isin(already_done)].copy()
print(f"Scraping {len(to_scrape)} remaining articles...")

results = []
fail_count = 0
success_count = 0

for i, (_, row) in enumerate(tqdm(
    to_scrape.iterrows(),
    total=len(to_scrape),
    desc="Scraping articles"
), start=1):

    text = fetch_text(row["url"])
    results.append({"story_id": row["story_id"], "text": text})

    if text:
        success_count += 1
    else:
        fail_count += 1

    if i % CACHE_SAVE_EVERY == 0:
        new_df = pd.DataFrame(results)
        cache_df = pd.concat([cache_df, new_df], ignore_index=True)
        cache_df.to_parquet(CACHE_PATH, index=False)
        results = []

    if i % LOG_EVERY == 0:
        rate = success_count / i
        remaining = len(to_scrape) - i
        print(
            f"\n[{i}/{len(to_scrape)}] "
            f"✓ {success_count} ok  ✗ {fail_count} failed  "
            f"({rate:.1%} success rate)  "
            f"~{remaining} left"
        )

    time.sleep(0.5)

# Final save
if results:
    new_df = pd.DataFrame(results)
    cache_df = pd.concat([cache_df, new_df], ignore_index=True)
    cache_df.to_parquet(CACHE_PATH, index=False)

# Merge texts into sampled_df
sampled_df = sampled_df.merge(cache_df[["story_id", "text"]], on="story_id", how="left")

print(f"\nDone. Success rate: {sampled_df['text'].notna().mean():.1%}")
print(f"Total fetched: {success_count}  Failed: {fail_count}")


Resuming: 27211 articles already cached
Scraping 3283 remaining articles...


Scraping articles:  15%|█▌        | 499/3283 [11:08<48:15,  1.04s/it]  


[500/3283] ✓ 302 ok  ✗ 198 failed  (60.4% success rate)  ~2783 left


Scraping articles:  28%|██▊       | 904/3283 [18:58<46:28,  1.17s/it]  Building prefix dict from c:\Users\annab\Documents\Work\2026_Senckenberg_Freelance\One_Health_Proposal_Eugenia\Mediacloud\Mediacloud_download+basic_graphs\.venv\Lib\site-packages\jieba\dict.txt ...
Dumping model to file cache C:\Users\annab\AppData\Local\Temp\jieba.cache
Loading model cost 1.1642775535583496 seconds.
Prefix dict has been built succesfully.
Scraping articles:  30%|███       | 999/3283 [20:45<37:34,  1.01it/s]  


[1000/3283] ✓ 709 ok  ✗ 291 failed  (70.9% success rate)  ~2283 left


...


[2500/3283] ✓ 1743 ok  ✗ 757 failed  (69.7% success rate)  ~783 left


Scraping articles:  91%|█████████▏| 2999/3283 [1:05:50<06:00,  1.27s/it]


[3000/3283] ✓ 2151 ok  ✗ 849 failed  (71.7% success rate)  ~283 left


Scraping articles: 100%|██████████| 3283/3283 [1:12:29<00:00,  1.32s/it]



Done. Success rate: 68.8%
Total fetched: 2385  Failed: 898
